In [0]:
import os
import json
import requests


def load_env(path=".env"):
    env = {}
    with open(path) as f:
        for line in f:
            line = line.strip()
            if not line or line.startswith("#") or "=" not in line:
                continue
            key, value = line.split("=", 1)
            env[key.strip()] = value.strip()
    return env


env = load_env()
API_KEY = env["FOOTBALL_API_KEY"]

BASE_URL = "https://v3.football.api-sports.io"
PREMIER_LEAGUE_ID = 39
SEASON = 2026  # api-sports uses the year the season starts, so 2026 = 2026/2027

headers = {"x-apisports-key": API_KEY}
params = {
    "search": "Palmer",
    "league": PREMIER_LEAGUE_ID,
    "season": SEASON,
}

response = requests.get(f"{BASE_URL}/players", headers=headers, params=params)
data = response.json()

print("Status code:", response.status_code)
print("Requests remaining today:", response.headers.get("x-ratelimit-requests-remaining"))
print(json.dumps(data, indent=2))

## Matchday player statistics

This pipeline now lives in `sync_matchday_stats.py` at the project root, so it
can run unattended on a schedule (see `com.footballdata.matchdaysync.plist`)
instead of only from this notebook.

It writes one CSV per finished matchday (`ENG_Premier_League_Matchday_NN.csv`)
and upserts each into a Google Drive folder by filename — so a Databricks
SCD Type 1/2 ingestion job always finds one file per matchday to diff
against. It's idempotent: a matchday is only re-fetched/re-uploaded if its
finished-fixture set grew, or it finished recently enough that api-football
might still be correcting stats.

Run it manually with:
```
python sync_matchday_stats.py
```

Required env vars (in `.env` or the shell environment):
- `FOOTBALL_API_KEY` (already set)
- `GOOGLE_DRIVE_FOLDER_ID` — the Drive folder to upload matchday CSVs into
- `GOOGLE_OAUTH_CLIENT_SECRET_FILE` — path to an OAuth "Desktop app" client
  secret JSON (defaults to `client_secret.json` in the project root)
- `GOOGLE_OAUTH_TOKEN_FILE` — where the saved refresh token lives after the
  first login (defaults to `token.json` in the project root)

One-time Google Drive setup (uploads happen as *your* Google account, since a
service account has no storage quota of its own and can't own files in a
normal Drive folder):
1. In Google Cloud Console, create an OAuth client ID of type "Desktop app"
   (enable the Drive API on the project first).
2. Download its JSON and save it as `client_secret.json` in the project root
   (already gitignored).
3. Copy your target Drive folder's ID (from its URL) into `GOOGLE_DRIVE_FOLDER_ID`.
4. Run `python sync_matchday_stats.py` once by hand — it opens a browser for
   you to log in and consent, then saves a refresh token to `token.json`.
   Every run after that (including scheduled/headless ones) refreshes
   silently with no browser prompt.

To automate on a schedule (macOS), install the launchd job:
```
cp com.footballdata.matchdaysync.plist ~/Library/LaunchAgents/
launchctl load ~/Library/LaunchAgents/com.footballdata.matchdaysync.plist
```
It runs every 3 hours and only does work when a new matchday is finished or
a recent one needs a stability re-check. Make sure you've done the one-time
interactive login above before loading it, since launchd can't complete a
browser consent flow.